In [37]:
!pip install gradio opencv-python matplotlib

In [46]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import gradio as gr

In [47]:
mineral_db = {

    'quartz': {
        'color': 'White/gray',
        'hardness': 7,
        'texture': 'Glassy, conchoidal fracture',
        'common_in': 'Veins, host rock',
        'hsv_range': ([0, 0, 180], [180, 30, 255])
    },

    'pyrite': {
        'color': 'Brassy yellow',
        'hardness': (6, 6.5),
        'texture': 'Cubic crystals, metallic luster',
        'common_in': 'Sulfides, igneous rock',
        'hsv_range': ([20, 100, 100], [40, 255, 255])
    },

    'biotite': {
        'color': 'Black/dark brown',
        'hardness': (2.5, 3),
        'texture': 'Platy, micaceous cleavage',
        'common_in': 'Igneous and metamorphic rock',
        'hsv_range': ([0, 0, 0], [180, 80, 90])
    },

    'bornite': {
        'color': 'Purplish/iridescent',
        'hardness': (3, 3.5),
        'texture': 'Metallic, tarnishes to purple',
        'common_in': 'Copper sulfide deposits',
        'hsv_range': ([120, 60, 60], [170, 255, 255])
    },

    'chrysocolla': {
        'color': 'Blue/green',
        'hardness': (2.5, 3.5),
        'texture': 'Waxy to earthy, amorphous',
        'common_in': 'Oxidized copper deposits',
        'hsv_range': ([75, 80, 80], [105, 255, 255])
    },

    'malachite': {
        'color': 'Bright green',
        'hardness': (3.5, 4),
        'texture': 'Banded, silky to vitreous',
        'common_in': 'Oxidized copper zones',
        'hsv_range': ([40, 50, 50], [85, 255, 255])
    },

    'muscovite': {
        'color': 'Colorless/pale silver',
        'hardness': (2, 2.5),
        'texture': 'Platy, pearly luster, flexible',
        'common_in': 'Granites, pegmatites, schists',
        'hsv_range': ([0, 0, 150], [180, 30, 220])
    }
}

In [48]:
def preprocess_image(image):

    # Resize image
    image = cv2.resize(image, (224, 224))

    # Reduce noise
    image = cv2.GaussianBlur(image, (5,5), 0)

    return image

In [49]:
def extract_features(image):

    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Ignore very dark pixels
    mask = hsv[:, :, 2] > 50

    if np.sum(mask) == 0:
        pixels = hsv.reshape(-1, 3)
    else:
        pixels = hsv[mask]

    # Mean HSV
    mean_hsv = np.mean(pixels, axis=0)

    # Standard deviation
    std_hsv = np.std(pixels, axis=0)

    # Brightness
    brightness = np.mean(gray)

    # Texture estimate using variance
    texture = np.var(gray)

    return {
        'mean_hsv': mean_hsv,
        'std_hsv': std_hsv,
        'brightness': brightness,
        'texture': texture
    }

In [50]:
def match_mineral(features):

    mean_hsv = features['mean_hsv']
    brightness = features['brightness']
    texture = features['texture']

    best_match = None
    best_score = float('inf')

    for mineral, props in mineral_db.items():

        low = np.array(props['hsv_range'][0], dtype=np.float32)
        high = np.array(props['hsv_range'][1], dtype=np.float32)

        # Center HSV
        center = (low + high) / 2

        # =========================
        # Core HSV Distance
        # =========================

        hsv_distance = np.linalg.norm(mean_hsv - center)

        # =========================
        # Saturation & Brightness
        # =========================

        saturation_penalty = abs(mean_hsv[1] - center[1]) * 0.3

        brightness_penalty = abs(mean_hsv[2] - center[2]) * 0.2

        # Base score
        score = (
            hsv_distance
            + saturation_penalty
            + brightness_penalty
        )

        # =========================
        # Mineral-specific Adjustments
        # =========================

        # Quartz (bright + low saturation)
        if mineral == "quartz":

            if brightness > 150:
                score -= 25

            if mean_hsv[1] < 40:
                score -= 15

        # Muscovite (light but slightly darker than quartz)
        if mineral == "muscovite":

            if 100 < brightness < 190:
                score -= 10

            if mean_hsv[1] < 50:
                score -= 8

        # Biotite (dark mineral)
        if mineral == "biotite":

            if brightness < 90:
                score -= 20

            if mean_hsv[1] < 80:
                score -= 10

        # Malachite (strong green)
        if mineral == "malachite":

            if 40 <= mean_hsv[0] <= 85:
                score -= 20

            if mean_hsv[1] > 100:
                score -= 10

        # Chrysocolla (blue-green/turquoise)
        if mineral == "chrysocolla":

            if 75 <= mean_hsv[0] <= 105:
                score -= 25

            if mean_hsv[1] > 80:
                score -= 10

        # Bornite (purple/iridescent)
        if mineral == "bornite":

            if 120 <= mean_hsv[0] <= 170:
                score -= 25

            if mean_hsv[1] > 60:
                score -= 8

        # Pyrite (gold/yellow metallic)
        if mineral == "pyrite":

            if 20 <= mean_hsv[0] <= 40:
                score -= 20

            if mean_hsv[1] > 100:
                score -= 10

        # =========================
        # Texture Influence
        # =========================

        # Highly textured minerals
        if mineral in ["pyrite", "bornite"]:

            if texture > 500:
                score -= 5

        # Smooth minerals
        if mineral in ["quartz", "muscovite"]:

            if texture < 300:
                score -= 5

        # =========================
        # Best Match Selection
        # =========================

        if score < best_score:

            best_score = score
            best_match = mineral

    # =========================
    # Confidence Calculation
    # =========================

    confidence = max(0, 100 - best_score)

    return best_match, round(confidence, 2)

In [51]:
def identify_mineral(image):

    # Convert RGB to BGR
    image_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

    # Preprocess image
    processed = preprocess_image(image_bgr)

    # Extract features
    features = extract_features(processed)

    # Match mineral
    mineral, confidence = match_mineral(features)

    # Get mineral info
    info = mineral_db[mineral]

    # Create output text
    result = f"""
===== Mineral Identification Result =====

Mineral: {mineral}

Confidence: {confidence:.2f}%

Color: {info['color']}

Hardness: {info['hardness']}

Texture: {info['texture']}

Common In: {info['common_in']}
"""

    return result

In [52]:
custom_css = """
body {
    background-color: #282828;
    color: #ebdbb2;
}

.gradio-container {
    background-color: #282828 !important;
    color: #ebdbb2 !important;
}

h1, h2, h3, p {
    color: #ebdbb2 !important;
}

button {
    background-color: #d79921 !important;
    color: #282828 !important;
    border: none !important;
}

button:hover {
    background-color: #fabd2f !important;
}

textarea {
    background-color: #3c3836 !important;
    color: #ebdbb2 !important;
}

input {
    background-color: #3c3836 !important;
    color: #ebdbb2 !important;
}
"""

In [53]:
iface = gr.Interface(
    fn=identify_mineral,

    inputs=gr.Image(
        type="numpy",
        label="Upload Mineral Image",
        height=500
    ),

    outputs=gr.Textbox(
        label="Identification Result",
        lines=14
    ),

    title="⛏️ Mineral Identification System",

    description="""
Upload a mineral image to identify the mineral type using HSV-based computer vision analysis.
""",

    css=custom_css,

    theme=gr.themes.Soft()
)

In [54]:
iface.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b4c006f8cdf960fbb2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 56, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://b4c006f8cdf960fbb2.gradio.live
